# AI6103 EfficientNetV2-S experiments on Kaggle

Attach two Kaggle inputs before running this notebook:

1. A dataset containing this project (`scripts/train.py` must be present).
2. The Dog Breed Identification data. Pre-split data may contain `train_labels.csv`, `val_labels.csv`, `test_labels.csv`, and matching `train/`, `val/`, `test/` image trees.

Enable a GPU under **Settings > Accelerator**. Run one 300-epoch configuration per notebook version. All generated files are written to `/kaggle/working/outputs`.

In [ ]:
# Experiment controls - edit these values before running.
EXPERIMENT = "smoke"  # smoke, lr_sweep, constant, cosine, wd_5e-4, wd_1e-4, mixup
BEST_LR = 0.005          # Replace after the learning-rate sweep.
BEST_WD = 0.0001        # Replace after comparing validation results.
RESUME = False

# Usually these can remain blank and will be discovered under /kaggle/input.
PROJECT_ROOT = ""
DATA_ROOT = ""

# To resume a prior Kaggle version, attach its output and give the directory
# containing the previous 'outputs' folder, for example:
# PREVIOUS_OUTPUTS = "/kaggle/input/ai6103-cosine-run/outputs"
PREVIOUS_OUTPUTS = ""

In [ ]:
from pathlib import Path
import csv
import json
import os
import shutil
import subprocess
import sys
import zipfile

import torch
import torchvision

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
OUTPUT_ROOT = WORKING_ROOT / "outputs"

def discover_project_root():
    matches = sorted(INPUT_ROOT.rglob("scripts/train.py"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one attached project containing scripts/train.py; found {matches}")
    return matches[0].parents[1]

def find_data_roots(search_root):
    candidate_roots = {p.parent for p in search_root.rglob("labels.csv")}
    candidate_roots.update(p.parent for p in search_root.rglob("train_labels.csv"))
    matches = []
    for root in sorted(candidate_roots):
        csv_stems = {p.stem.lower() for p in root.glob("*.csv")}
        has_single_labels = "labels" in csv_stems
        has_split_labels = {"train_labels", "test_labels"}.issubset(csv_stems) and bool({"val_labels", "validation_labels"} & csv_stems)
        image_count = sum(1 for p in root.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
        if (has_single_labels or has_split_labels) and image_count >= 10_222:
            matches.append((root, image_count, sorted(csv_stems)))
    return matches

def discover_data_root():
    matches = find_data_roots(INPUT_ROOT)
    if not matches:
        data_archives = []
        for archive in INPUT_ROOT.rglob("*.zip"):
            with zipfile.ZipFile(archive) as handle:
                stems = {Path(name).stem.lower() for name in handle.namelist()}
            has_split_labels = {"train_labels", "test_labels"}.issubset(stems) and bool({"val_labels", "validation_labels"} & stems)
            if "labels" in stems or has_split_labels:
                data_archives.append(archive)
        if len(data_archives) != 1:
            raise RuntimeError(f"Could not uniquely identify the uploaded data ZIP; candidates={data_archives}")
        extraction_root = WORKING_ROOT / "extracted_data"
        extraction_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(data_archives[0]) as handle:
            destination = extraction_root.resolve()
            for member in handle.infolist():
                member_path = (extraction_root / member.filename).resolve()
                if destination not in member_path.parents and member_path != destination:
                    raise RuntimeError(f"Unsafe path in ZIP: {member.filename}")
            handle.extractall(extraction_root)
        print(f"Extracted {data_archives[0]} to {extraction_root}")
        matches = find_data_roots(extraction_root)
    if len(matches) != 1:
        raise RuntimeError(f"Expected one data root with label CSVs and at least 10,222 images; found {matches}")
    return matches[0][0]

project_root = Path(PROJECT_ROOT) if PROJECT_ROOT else discover_project_root()
data_root = Path(DATA_ROOT) if DATA_ROOT else discover_data_root()

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU is not enabled. Select a GPU accelerator in Kaggle settings.")
print("Project root:", project_root)
print("Data root:", data_root)
print("Output root:", OUTPUT_ROOT)

In [ ]:
# Validate the complete dataset layout before starting a GPU training process.
from collections import Counter

source_root = str(project_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)
from ai6103.data import create_or_load_manifest

label_files = sorted(data_root.rglob("*.csv"))
print("CSV files discovered:")
for label_file in label_files:
    with label_file.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.reader(handle)
        header = next(reader, [])
        row_count = sum(1 for _ in reader)
    print(f"  {label_file.relative_to(data_root)}: rows={row_count}, columns={header}")

preflight_manifest = OUTPUT_ROOT / "data_artifacts" / "split_manifest.csv"
preflight_records = create_or_load_manifest(data_root, preflight_manifest, seed=6103)
print("Validated split counts:", dict(Counter(record.split for record in preflight_records)))
print("Class count:", len({record.breed for record in preflight_records}))
print("Dataset preflight passed.")

In [ ]:
# Optional recovery: copy an attached previous output into writable storage.
if PREVIOUS_OUTPUTS:
    previous = Path(PREVIOUS_OUTPUTS)
    if not previous.is_dir():
        raise FileNotFoundError(previous)
    shutil.copytree(previous, OUTPUT_ROOT, dirs_exist_ok=True)
    print(f"Recovered previous outputs from {previous}")
elif RESUME:
    print("RESUME=True without PREVIOUS_OUTPUTS: expecting checkpoints already in /kaggle/working/outputs.")

In [ ]:
def run_training(name, learning_rate, epochs, scheduler="none", weight_decay=0.0, mixup_alpha=0.0):
    output_dir = OUTPUT_ROOT / name
    command = [
        sys.executable, str(project_root / "scripts" / "train.py"),
        "--data-root", str(data_root),
        "--output-dir", str(output_dir),
        "--lr", str(learning_rate),
        "--epochs", str(epochs),
        "--scheduler", scheduler,
        "--weight-decay", str(weight_decay),
        "--mixup-alpha", str(mixup_alpha),
        "--batch-size", "128",
        "--momentum", "0.9",
        "--seed", "6103",
        "--num-workers", "4",
        "--device", "cuda",
    ]
    if RESUME:
        command.append("--resume")
    print("Running:", " ".join(command))
    subprocess.run(command, check=True, cwd=project_root)
    return output_dir

EXPERIMENTS = {
    "smoke": [("smoke/lr_0.005", 0.005, 1, "none", 0.0, 0.0)],
    "lr_sweep": [
        ("lr/lr_0.5", 0.5, 15, "none", 0.0, 0.0),
        ("lr/lr_0.005", 0.005, 15, "none", 0.0, 0.0),
        ("lr/lr_0.001", 0.001, 15, "none", 0.0, 0.0),
    ],
    "constant": [("schedule/constant", BEST_LR, 300, "none", 0.0, 0.0)],
    "cosine": [("schedule/cosine", BEST_LR, 300, "cosine", 0.0, 0.0)],
    "wd_5e-4": [("weight_decay/wd_5e-4", BEST_LR, 300, "cosine", 5e-4, 0.0)],
    "wd_1e-4": [("weight_decay/wd_1e-4", BEST_LR, 300, "cosine", 1e-4, 0.0)],
    "mixup": [("mixup/alpha_0.2", BEST_LR, 300, "cosine", BEST_WD, 0.2)],
}

if EXPERIMENT not in EXPERIMENTS:
    raise ValueError(f"Unknown experiment: {EXPERIMENT}")
completed_runs = [run_training(*spec) for spec in EXPERIMENTS[EXPERIMENT]]
completed_runs

In [ ]:
# Display final numerical results and generated training curves.
from IPython.display import Image, display

for run_dir in completed_runs:
    summary_path = run_dir / "summary.json"
    summary = json.loads(summary_path.read_text())
    print(f"\n{run_dir.relative_to(OUTPUT_ROOT)}")
    print(json.dumps(summary["final"], indent=2))
    display(Image(filename=str(run_dir / "training_curves.png")))

In [ ]:
# Package outputs for download or save them as this notebook version's output.
archive = shutil.make_archive("/kaggle/working/ai6103_outputs", "zip", OUTPUT_ROOT)
print("Created:", archive)
print("Save a notebook version with outputs enabled before ending the session.")

## Final test evaluation

Do **not** evaluate the test split during tuning. After selecting the final configuration using validation results, run the separate command below once, replacing the checkpoint path:

```python
subprocess.run([
    sys.executable, str(project_root / 'scripts' / 'evaluate_test.py'),
    '--checkpoint', '/kaggle/working/outputs/mixup/alpha_0.2/best_checkpoint.pt',
    '--data-root', str(data_root),
    '--output', '/kaggle/working/outputs/final_test.json',
    '--device', 'cuda',
], check=True)
```